In [8]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Tuple, Sequence, List
from functools import lru_cache
import math


@dataclass(frozen=True)
class FingeringState:
    """
    fret: 固定长度4，例如 [0,2,3,1]
    strings: 当前被弹到的弦，长度 1~4，例如 [0,3] / [0,1,2,3]
    """
    fret: Tuple[int, int, int, int]
    strings: Tuple[int, ...]

    @staticmethod
    def from_lists(fret: Sequence[int], strings: Sequence[int]) -> "FingeringState":
        if len(fret) != 4:
            raise ValueError("fret 必须固定为长度 4")

        if not (1 <= len(strings) <= 4):
            raise ValueError("strings 的长度必须在 1~4 之间")

        fret = tuple(int(x) for x in fret)
        strings = tuple(sorted(set(int(s) for s in strings)))

        if len(strings) == 0 or len(strings) > 4:
            raise ValueError("strings 去重后长度必须在 1~4 之间")

        for f in fret:
            if f < 0:
                raise ValueError("fret 不能为负数")

        for s in strings:
            if s not in (0, 1, 2, 3):
                raise ValueError("strings 中元素必须在 0~3 之间")

        return FingeringState(fret=fret, strings=strings)


def active_positions(
    state: FingeringState,
    only_played_strings: bool = False
) -> List[Tuple[int, int]]:
    """
    从 fret 中提取当前左手 active positions: [(string, fret), ...]

    默认 only_played_strings=False:
        只要 fret > 0，就认为左手在按，参与 transition/state cost。
        这通常更符合“左手真实姿态”。

    如果 only_played_strings=True:
        只有 fret > 0 且 string 在 state.strings 里的位置才参与。
        这更接近“只对当前发声事件计成本”。
    """
    played = set(state.strings)
    pos = []
    for s, f in enumerate(state.fret):
        if f > 0:
            if (not only_played_strings) or (s in played):
                pos.append((s, f))
    return pos


def state_cost(
    state: FingeringState,
    w3: float,
    w4: float,
    w5: float,
    gamma: float,
    only_played_strings: bool = False
) -> float:
    """
    你的静态代价：
        w3 * log(1 + max(0, avg_active_fret - gamma))
      + w4 * count(nonzero fret)
      + w5 * fret_span
    """
    pos = active_positions(state, only_played_strings=only_played_strings)
    active_frets = [f for _, f in pos]

    if not active_frets:
        avg_active_fret = 0.0
        fret_span = 0.0
        count_nonzero = 0
    else:
        avg_active_fret = sum(active_frets) / len(active_frets)
        fret_span = max(active_frets) - min(active_frets)
        count_nonzero = len(active_frets)

    return (
        w3 * math.log(1.0 + max(0.0, avg_active_fret - gamma))
        + w4 * count_nonzero
        + w5 * fret_span
    )


def transition_cost(
    prev_state: FingeringState,
    curr_state: FingeringState,
    w0: float,   # 新增手指 cost
    w1: float,   # 弦距离权重
    w2: float,   # 品距离权重
    only_played_strings: bool = False
) -> float:
    """
    局部两帧之间的“整体最小多对多匹配代价”。

    匹配对象:
        prev_active_positions  <->  curr_active_positions

    匹配代价:
        old -> new : w1*|s2-s1| + w2*|f2-f1|
        new unmatched: w0
        old unmatched: 0   (释放手指免费)

    注意:
    - 这是局部 assignment / matching
    - 不是整首歌的图搜索
    """
    prev_pos = active_positions(prev_state, only_played_strings=only_played_strings)
    curr_pos = active_positions(curr_state, only_played_strings=only_played_strings)

    m = len(prev_pos)
    n = len(curr_pos)

    @lru_cache(maxsize=None)
    def dp(j: int, used_mask: int) -> float:
        if j == n:
            # curr 全处理完，prev 剩余未匹配的都是 release，代价 0
            return 0.0

        curr_s, curr_f = curr_pos[j]

        # 方案1：curr 这个位置是“新增手指”
        best = w0 + dp(j + 1, used_mask)

        # 方案2：匹配到某个尚未用过的 prev 位置
        for i in range(m):
            if (used_mask >> i) & 1:
                continue
            prev_s, prev_f = prev_pos[i]
            move_cost = w1 * abs(curr_s - prev_s) + w2 * abs(curr_f - prev_f)
            cand = move_cost + dp(j + 1, used_mask | (1 << i))
            if cand < best:
                best = cand

        return best

    return dp(0, 0)


def full_step_cost(
    prev_state: FingeringState,
    curr_state: FingeringState,
    w0: float,
    w1: float,
    w2: float,
    w3: float,
    w4: float,
    w5: float,
    gamma: float,
    only_played_strings: bool = False
) -> float:
    return (
        transition_cost(
            prev_state, curr_state,
            w0=w0, w1=w1, w2=w2,
            only_played_strings=only_played_strings
        )
        + state_cost(
            curr_state,
            w3=w3, w4=w4, w5=w5, gamma=gamma,
            only_played_strings=only_played_strings
        )
    )

In [ ]:
import math


def approx_equal(a: float, b: float, eps: float = 1e-9) -> bool:
    return abs(a - b) < eps


def run_tests():
    print("Running tests...")

    # -------------------------
    # Test 1: 状态构造
    # -------------------------
    s = FingeringState.from_lists(
        fret=[0, 2, 3, 1],
        strings=[0, 3]
    )
    assert s.fret == (0, 2, 3, 1)
    assert s.strings == (0, 3)
    print("Test 1 passed: FingeringState.from_lists")

    # -------------------------
    # Test 2: active_positions
    # -------------------------
    pos_all = active_positions(s, only_played_strings=False)
    pos_played = active_positions(s, only_played_strings=True)

    assert pos_all == [(1, 2), (2, 3), (3, 1)]
    assert pos_played == [(3, 1)]
    print("Test 2 passed: active_positions")

    # -------------------------
    # Test 3: state_cost
    # 用容易手算的权重
    # -------------------------
    # s = fret [0,2,3,1]
    # active frets = [2,3,1]
    # avg = 2
    # count = 3
    # span = 2
    #
    # gamma = 2
    # log(1 + max(0, 2-2)) = log(1)=0
    #
    # cost = w3*0 + w4*3 + w5*2
    #      = 1*0 + 1*3 + 1*2 = 5
    c1 = state_cost(
        s,
        w3=1.0,
        w4=1.0,
        w5=1.0,
        gamma=2.0,
        only_played_strings=False
    )
    assert approx_equal(c1, 5.0)

    # only_played_strings=True 时，只保留 3弦1品
    # active frets = [1]
    # avg = 1, count = 1, span = 0
    # gamma = 2 -> log(1+max(0,1-2)) = 0
    # cost = 1
    c2 = state_cost(
        s,
        w3=1.0,
        w4=1.0,
        w5=1.0,
        gamma=2.0,
        only_played_strings=True
    )
    assert approx_equal(c2, 1.0)
    print("Test 3 passed: state_cost")

    # -------------------------
    # Test 4: transition_cost
    # 验证“整体最优匹配”，不是逐弦对位
    # -------------------------
    prev_state = FingeringState.from_lists(
        fret=[0, 2, 3, 1],   # active = (1,2), (2,3), (3,1)
        strings=[0, 1, 2, 3]
    )
    curr_state = FingeringState.from_lists(
        fret=[1, 0, 2, 3],   # active = (0,1), (2,2), (3,3)
        strings=[0, 1, 2, 3]
    )

    # 设 w0=10, w1=1, w2=1
    # 一种最优匹配：
    # (1,2) -> (0,1): |1-0| + |2-1| = 2
    # (2,3) -> (2,2): |2-2| + |3-2| = 1
    # (3,1) -> (3,3): |3-3| + |1-3| = 2
    # 总和 = 5
    tc = transition_cost(
        prev_state,
        curr_state,
        w0=10.0,
        w1=1.0,
        w2=1.0,
        only_played_strings=False
    )
    assert approx_equal(tc, 5.0)
    print("Test 4 passed: transition_cost global matching")

    # -------------------------
    # Test 5: 新增手指 cost
    # -------------------------
    prev_state = FingeringState.from_lists(
        fret=[0, 2, 0, 0],   # active = (1,2)
        strings=[1]
    )
    curr_state = FingeringState.from_lists(
        fret=[0, 2, 3, 0],   # active = (1,2), (2,3)
        strings=[1, 2]
    )

    # 最优：
    # (1,2)->(1,2) = 0
    # 新增 (2,3) = w0 = 10
    tc = transition_cost(
        prev_state,
        curr_state,
        w0=10.0,
        w1=1.0,
        w2=1.0,
        only_played_strings=False
    )
    assert approx_equal(tc, 10.0)
    print("Test 5 passed: new finger cost")

    # -------------------------
    # Test 6: 释放手指免费
    # -------------------------
    prev_state = FingeringState.from_lists(
        fret=[0, 2, 3, 0],   # active = (1,2), (2,3)
        strings=[1, 2]
    )
    curr_state = FingeringState.from_lists(
        fret=[0, 2, 0, 0],   # active = (1,2)
        strings=[1]
    )

    # 最优：
    # (1,2)->(1,2) = 0
    # (2,3) 释放 = 0
    tc = transition_cost(
        prev_state,
        curr_state,
        w0=10.0,
        w1=1.0,
        w2=1.0,
        only_played_strings=False
    )
    assert approx_equal(tc, 0.0)
    print("Test 6 passed: release is free")

    # -------------------------
    # Test 7: 跨弦移动
    # -------------------------
    prev_state = FingeringState.from_lists(
        fret=[0, 2, 0, 0],   # active = (1,2)
        strings=[1]
    )
    curr_state = FingeringState.from_lists(
        fret=[2, 0, 0, 0],   # active = (0,2)
        strings=[0]
    )

    # (1,2)->(0,2): |1-0| + |2-2| = 1
    tc = transition_cost(
        prev_state,
        curr_state,
        w0=10.0,
        w1=1.0,
        w2=1.0,
        only_played_strings=False
    )
    assert approx_equal(tc, 1.0)
    print("Test 7 passed: cross-string move")

    # -------------------------
    # Test 8: only_played_strings=True 对 transition 的影响
    # -------------------------
    prev_state = FingeringState.from_lists(
        fret=[0, 1, 2, 2],
        strings=[1,2,3]
    )
    curr_state = FingeringState.from_lists(
        fret=[2, 3, 2, 0],
        strings=[0, 1, 2]
    )

    # only_played_strings=True:
    # prev active -> 只有 (3,1)
    # curr active -> 只有 (3,1)
    # 所以 transition = 0
    tc = transition_cost(
        prev_state,
        curr_state,
        w0=2.0,
        w1=2.0,
        w2=1.0,
        only_played_strings=True
    )
    print(f"{tc = }")


if __name__ == "__main__":
    run_tests()

Running tests...
Test 1 passed: FingeringState.from_lists
Test 2 passed: active_positions
Test 3 passed: state_cost
Test 4 passed: transition_cost global matching
Test 5 passed: new finger cost
Test 6 passed: release is free
Test 7 passed: cross-string move
tc = 4.0


AssertionError: 

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Sequence, Tuple, List, Dict
import json
import random
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------
# 1) Your cost design (same idea)
# -----------------------------
@dataclass(frozen=True)
class FingeringState:
    fret: Tuple[int, int, int, int]
    strings: Tuple[int, ...]

    @staticmethod
    def from_lists(fret: Sequence[int], strings: Sequence[int]) -> "FingeringState":
        if len(fret) != 4:
            raise ValueError("fret must be length 4")
        fret = tuple(int(x) for x in fret)
        strings = tuple(sorted(set(int(s) for s in strings)))
        if not (1 <= len(strings) <= 4):
            raise ValueError("strings length must be in [1, 4]")
        for f in fret:
            if f < 0:
                raise ValueError("fret cannot be negative")
        for s in strings:
            if s not in (0, 1, 2, 3):
                raise ValueError("string index must be 0..3")
        return FingeringState(fret=fret, strings=strings)


def active_positions(state: FingeringState, only_played_strings: bool = False) -> List[Tuple[int, int]]:
    played = set(state.strings)
    pos = []
    for s, f in enumerate(state.fret):
        if f > 0 and ((not only_played_strings) or (s in played)):
            pos.append((s, f))
    return pos


def state_cost_torch(
    state: FingeringState,
    w3: torch.Tensor,
    w4: torch.Tensor,
    w5: torch.Tensor,
    gamma: torch.Tensor,
    device: torch.device,
    only_played_strings: bool = False,
) -> torch.Tensor:
    pos = active_positions(state, only_played_strings=only_played_strings)
    if not pos:
        avg_active_fret = torch.tensor(0.0, device=device)
        fret_span = torch.tensor(0.0, device=device)
        count_nonzero = torch.tensor(0.0, device=device)
    else:
        active_frets = torch.tensor([f for _, f in pos], dtype=torch.float32, device=device)
        avg_active_fret = active_frets.mean()
        fret_span = active_frets.max() - active_frets.min()
        count_nonzero = torch.tensor(float(len(pos)), dtype=torch.float32, device=device)

    return (
        w3 * torch.log1p(torch.relu(avg_active_fret - gamma))
        + w4 * count_nonzero
        + w5 * fret_span
    )


def transition_cost_torch(
    prev_state,
    curr_state,
    w0: torch.Tensor,   # 新增手指 cost
    w1: torch.Tensor,   # 弦移动 cost
    w2: torch.Tensor,   # 品移动 cost
    device: torch.device,
    only_played_strings: bool = False,
) -> torch.Tensor:
    """
    计算相邻两帧 prev_state -> curr_state 的最小整体转换代价。

    代价定义:
        1) 旧位置 prev_pos[i] 匹配到新位置 curr_pos[j]:
           cost = w1 * |curr_s - prev_s| + w2 * |curr_f - prev_f|

        2) curr_pos[j] 没有任何旧位置匹配到:
           cost = w0   (新增手指)

        3) prev_pos[i] 没有匹配到任何新位置:
           cost = 0    (释放手指免费)

    说明:
    - 这是“局部两帧之间”的最优 assignment / matching cost
    - 不是整条序列 across time 的全局图搜索
    - 这个函数返回的结果可以作为外层 time-layer DP / shortest path 的边权

    要求:
    - 你已经定义好了 active_positions(state, only_played_strings=False)
      并返回形如 [(string, fret), ...] 的列表
    """

    prev_pos = active_positions(prev_state, only_played_strings=only_played_strings)
    curr_pos = active_positions(curr_state, only_played_strings=only_played_strings)

    m = len(prev_pos)   # 上一帧 active positions 数量
    n = len(curr_pos)   # 当前帧 active positions 数量

    dtype = w0.dtype
    inf = torch.tensor(1e9, device=device, dtype=dtype)
    zero = torch.tensor(0.0, device=device, dtype=dtype)

    # 边界情况
    if n == 0:
        # 当前没有任何按法，全部释放，代价 0
        return zero

    if m == 0:
        # 上一帧没有任何按法，当前全部都是新增
        return w0 * n

    # 预先计算所有 1->1 匹配代价矩阵 cost[i][j]
    # i: prev index, j: curr index
    pair_cost = []
    for i in range(m):
        prev_s, prev_f = prev_pos[i]
        row = []
        for j in range(n):
            curr_s, curr_f = curr_pos[j]
            move = (
                w1 * abs(curr_s - prev_s)
                + w2 * abs(curr_f - prev_f)
            )
            row.append(move)
        pair_cost.append(row)

    # 子集 DP
    # dp[mask] = 处理完当前若干个 curr 后，已经使用了 prev 的 mask 时的最小代价
    # 初始时，什么都没匹配，代价为 0
    dp = {0: zero}
    
    # 逐个处理 curr_pos[j]
    for j in range(n):
        nxt = {}

        for mask, base_cost in dp.items():
            # 方案 1: curr_pos[j] 作为“新增手指”
            cand_new = base_cost + w0
            if mask not in nxt:
                nxt[mask] = cand_new
            else:
                nxt[mask] = torch.minimum(nxt[mask], cand_new)

            # 方案 2: 匹配到某个还没被使用过的 prev_pos[i]
            for i in range(m):
                if (mask >> i) & 1:
                    continue  # 这个 prev 已经被占用了

                new_mask = mask | (1 << i)
                cand_match = base_cost + pair_cost[i][j]

                if new_mask not in nxt:
                    nxt[new_mask] = cand_match
                else:
                    nxt[new_mask] = torch.minimum(nxt[new_mask], cand_match)

        dp = nxt

    # 所有 curr 都处理完后，剩余未匹配的 prev 自动释放，代价 0
    # 所以直接在所有终止 mask 中取最小值
    best = inf
    for v in dp.values():
        best = torch.minimum(best, v)

    return best


def full_song_cost_torch(
    states: List[FingeringState],
    w0: torch.Tensor,
    w1: torch.Tensor,
    w2: torch.Tensor,
    w3: torch.Tensor,
    w4: torch.Tensor,
    w5: torch.Tensor,
    gamma: torch.Tensor,
    device: torch.device,
    only_played_strings: bool = False,
) -> torch.Tensor:
    if not states:
        return torch.tensor(0.0, device=device)

    total = state_cost_torch(states[0], w3, w4, w5, gamma, device, only_played_strings)
    for i in range(1, len(states)):
        total = total + transition_cost_torch(states[i - 1], states[i], w0, w1, w2, device, only_played_strings)
        total = total + state_cost_torch(states[i], w3, w4, w5, gamma, device, only_played_strings)

    return total / len(states)  # normalize by song length


def per_step_costs_torch(
    states: List[FingeringState],
    w0: torch.Tensor,
    w1: torch.Tensor,
    w2: torch.Tensor,
    w3: torch.Tensor,
    w4: torch.Tensor,
    w5: torch.Tensor,
    gamma: torch.Tensor,
    device: torch.device,
    only_played_strings: bool = False,
) -> torch.Tensor:
    """与 full_song_cost_torch 一致：第 i 步代价，使得 mean 等于整首歌平均代价。"""
    if not states:
        return torch.tensor([], device=device, dtype=torch.float32)

    costs = []
    c0 = state_cost_torch(states[0], w3, w4, w5, gamma, device, only_played_strings)
    costs.append(c0)
    for i in range(1, len(states)):
        t = transition_cost_torch(
            states[i - 1], states[i], w0, w1, w2, device, only_played_strings
        )
        s = state_cost_torch(states[i], w3, w4, w5, gamma, device, only_played_strings)
        costs.append(t + s)
    return torch.stack(costs)


def predict_per_note_difficulties(
    model: "WeightNet",
    states: List[FingeringState],
    device: torch.device,
    only_played_strings: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    返回整首歌预测难度 pred，以及逐帧 difficulty（与 pred 同量纲，按逐步代价占比分配，均值等于 pred）。
    """
    if not states:
        z = torch.tensor(0.0, device=device)
        return z, torch.tensor([], device=device)

    feat = song_features(states).to(device)
    h = model.backbone(feat)
    raw = model.param_head(h)
    w = F.softplus(raw[:6]) + 1e-4
    gamma = F.softplus(raw[6])
    step_costs = per_step_costs_torch(
        states,
        w0=w[0],
        w1=w[1],
        w2=w[2],
        w3=w[3],
        w4=w[4],
        w5=w[5],
        gamma=gamma,
        device=device,
        only_played_strings=only_played_strings,
    )
    song_cost = step_costs.mean()
    pred = model.diff_head(torch.cat([h, song_cost.view(1)], dim=0)).squeeze(0)
    mean_c = step_costs.mean().clamp(min=1e-8)
    per_note = pred * (step_costs / mean_c)
    return pred, per_note


def save_weightnet_checkpoint(
    model: "WeightNet",
    path: Path,
    *,
    only_played_strings: bool,
    in_dim: int = 8,
    hidden: int = 32,
) -> None:
    torch.save(
        {
            "state_dict": model.state_dict(),
            "only_played_strings": only_played_strings,
            "in_dim": in_dim,
            "hidden": hidden,
        },
        path,
    )


def load_weightnet(
    ckpt_path: Path | str,
    map_location: torch.device | str | None = None,
) -> Tuple["WeightNet", torch.device, bool]:
    device = map_location or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
        in_dim = int(ckpt.get("in_dim", 8))
        hidden = int(ckpt.get("hidden", 32))
        only_played = bool(ckpt.get("only_played_strings", False))
    else:
        state_dict = ckpt
        in_dim, hidden = 8, 32
        only_played = False
    model = WeightNet(in_dim=in_dim, hidden=hidden).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, device, only_played


def difficulty_from_frets_strings(
    frets: List[List[int]],
    strings: List[List[int]],
    ckpt_path: Path | str | None = None,
    *,
    only_played_strings: bool | None = None,
) -> Tuple[float, List[float]]:
    """
    输入与 JSON 中相同的逐帧 fret（长度4）与 strings，返回 (整首歌难度预测, 逐帧 difficult_level)。

    ckpt_path 默认：当前目录或 song_info_fret_strings_map.json 同目录下的 weightnet_song_difficulty.pt。
    """
    path = Path(ckpt_path) if ckpt_path is not None else Path("weightnet_song_difficulty.pt")
    if not path.is_file():
        m = Path("song_info_fret_strings_map.json")
        if m.resolve().is_file():
            path = m.resolve().parent / "weightnet_song_difficulty.pt"
    if not path.is_file():
        raise FileNotFoundError(f"找不到 checkpoint: {path}，请先训练保存或传入 ckpt_path")
    model, device, ops_default = load_weightnet(path)
    ops = ops_default if only_played_strings is None else only_played_strings

    states = build_song_states({"fret": frets, "strings": strings})
    if not states:
        return 0.0, []

    pred, per = predict_per_note_difficulties(model, states, device, only_played_strings=ops)
    return float(pred.detach().cpu()), [float(x) for x in per.detach().cpu()]


# -----------------------------
# 2) Data loading
# -----------------------------
def build_song_states(entry: Dict) -> List[FingeringState]:
    frets = entry.get("fret", [])
    strings = entry.get("strings", [])
    n = min(len(frets), len(strings))
    out = []
    for i in range(n):
        try:
            out.append(FingeringState.from_lists(frets[i], strings[i]))
        except Exception:
            continue
    return out


def song_features(states: List[FingeringState]) -> torch.Tensor:
    # simple hand-crafted song-level features
    if not states:
        return torch.zeros(8, dtype=torch.float32)

    n_notes = len(states)
    nonzero_counts = []
    mean_frets = []
    spans = []
    played_counts = []

    for st in states:
        nz = [x for x in st.fret if x > 0]
        nonzero_counts.append(len(nz))
        mean_frets.append((sum(nz) / len(nz)) if nz else 0.0)
        spans.append((max(nz) - min(nz)) if len(nz) > 1 else 0.0)
        played_counts.append(len(st.strings))

    feat = torch.tensor([
        float(n_notes),
        float(sum(nonzero_counts) / n_notes),
        float(sum(mean_frets) / n_notes),
        float(sum(spans) / n_notes),
        float(sum(played_counts) / n_notes),
        float(max(nonzero_counts)),
        float(max(mean_frets)),
        float(max(spans)),
    ], dtype=torch.float32)

    feat[0] = torch.log1p(feat[0])  # mild scaling
    return feat


# -----------------------------
# 3) Model: dynamic weight predictor
# -----------------------------
class WeightNet(nn.Module):
    def __init__(self, in_dim: int = 8, hidden: int = 32):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.param_head = nn.Linear(hidden, 7)  # w0..w5 + gamma
        self.diff_head = nn.Linear(hidden + 1, 1)  # [context, song_cost] -> difficulty

    def forward(self, feat: torch.Tensor, song_cost: torch.Tensor):
        h = self.backbone(feat)
        raw = self.param_head(h)
        w = F.softplus(raw[:6]) + 1e-4
        gamma = F.softplus(raw[6])
        x = torch.cat([h, song_cost.view(1)], dim=0)
        pred = self.diff_head(x).squeeze(0)
        return pred, w, gamma


# -----------------------------
# 4) Train / eval
# -----------------------------
def run():
    random.seed(42)
    torch.manual_seed(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    map_path = Path("song_info_fret_strings_map.json")
    if not map_path.exists():
        raise FileNotFoundError(f"Cannot find {map_path.resolve()}")

    with map_path.open("r", encoding="utf-8") as f:
        song_map = json.load(f)

    samples = []
    for key, entry in song_map.items():
        diff = entry.get("difficulty_level", None)
        if diff is None:
            continue
        states = build_song_states(entry)
        if len(states) < 2:
            continue
        feat = song_features(states)
        samples.append((key, states, feat, float(diff)))

    print("usable songs:", len(samples))
    random.shuffle(samples)

    split = int(len(samples) * 0.8)
    train_samples = samples[:split]
    val_samples = samples[split:]
    print("train:", len(train_samples), "val:", len(val_samples))

    model = WeightNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    only_played_strings = False
    epochs = 40

    def epoch_step(data, train=True):
        if train:
            model.train()
        else:
            model.eval()

        total_loss, total_mae, n = 0.0, 0.0, 0
        for key, states, feat, y in data:
            feat = feat.to(device)
            y_t = torch.tensor(y, dtype=torch.float32, device=device)

            if train:
                opt.zero_grad()

            h = model.backbone(feat)
            raw = model.param_head(h)
            w = F.softplus(raw[:6]) + 1e-4
            gamma = F.softplus(raw[6])

            song_cost = full_song_cost_torch(
                states,
                w0=w[0], w1=w[1], w2=w[2],
                w3=w[3], w4=w[4], w5=w[5],
                gamma=gamma,
                device=device,
                only_played_strings=only_played_strings,
            )

            pred = model.diff_head(torch.cat([h, song_cost.view(1)], dim=0)).squeeze(0)

            loss = F.mse_loss(pred, y_t)
            reg = 1e-4 * (w.pow(2).sum() + gamma.pow(2))
            loss = loss + reg

            if train:
                loss.backward()
                opt.step()

            total_loss += float(loss.detach().cpu())
            total_mae += float((pred.detach() - y_t).abs().cpu())
            n += 1

        return total_loss / max(n, 1), total_mae / max(n, 1)

    for ep in range(1, epochs + 1):
        tr_loss, tr_mae = epoch_step(train_samples, train=True)
        va_loss, va_mae = epoch_step(val_samples, train=False)
        if ep == 1 or ep % 5 == 0:
            print(
                f"epoch={ep:03d} | "
                f"train loss={tr_loss:.4f}, mae={tr_mae:.4f} | "
                f"val loss={va_loss:.4f}, mae={va_mae:.4f}"
            )

    # show examples
    model.eval()
    print("\n--- validation examples ---")
    for i in range(min(8, len(val_samples))):
        key, states, feat, y = val_samples[i]
        feat = feat.to(device)

        h = model.backbone(feat)
        raw = model.param_head(h)
        w = F.softplus(raw[:6]) + 1e-4
        gamma = F.softplus(raw[6])

        song_cost = full_song_cost_torch(
            states,
            w0=w[0], w1=w[1], w2=w[2],
            w3=w[3], w4=w[4], w5=w[5],
            gamma=gamma,
            device=device,
            only_played_strings=only_played_strings,
        )
        pred = model.diff_head(torch.cat([h, song_cost.view(1)], dim=0)).squeeze(0)

        print(f"{key} | y={y:.1f} pred={float(pred.cpu()):.3f} cost={float(song_cost.cpu()):.3f}")
        print("  w0..w5:", [round(float(x), 4) for x in w.detach().cpu()])
        print("  gamma:", round(float(gamma.detach().cpu()), 4))

    # save model（含训练配置，便于下次 load_weightnet / difficulty_from_frets_strings）
    save_path = map_path.resolve().parent / "weightnet_song_difficulty.pt"
    save_weightnet_checkpoint(
        model,
        save_path,
        only_played_strings=only_played_strings,
        in_dim=8,
        hidden=32,
    )
    print("\nmodel saved to:", save_path.resolve())


if __name__ == "__main__":
    run()

device: cuda
usable songs: 888
train: 710 val: 178
epoch=001 | train loss=4.0563, mae=1.5802 | val loss=3.2900, mae=1.4247
epoch=005 | train loss=2.7140, mae=1.3183 | val loss=3.0047, mae=1.3572
epoch=010 | train loss=2.5857, mae=1.2760 | val loss=2.8420, mae=1.3024
epoch=015 | train loss=2.5067, mae=1.2525 | val loss=2.7947, mae=1.2737
epoch=020 | train loss=2.4528, mae=1.2325 | val loss=2.7701, mae=1.2566
epoch=025 | train loss=2.4092, mae=1.2155 | val loss=2.7587, mae=1.2470
epoch=030 | train loss=2.3787, mae=1.2075 | val loss=2.7104, mae=1.2291
epoch=035 | train loss=2.3487, mae=1.1968 | val loss=2.7024, mae=1.2223
epoch=040 | train loss=2.3288, mae=1.1902 | val loss=2.6783, mae=1.2151

--- validation examples ---
smoWxZ_eajFTc | y=5.0 pred=4.352 cost=5.592
  w0..w5: [1.6421, 1.0863, 1.8968, 0.1837, 1.1144, 2.2474]
  gamma: 0.1269
s78K4I_e2NFFe | y=8.0 pred=6.112 cost=8.181
  w0..w5: [1.3579, 0.8896, 3.3779, 0.1376, 1.1343, 2.8065]
  gamma: 0.0323
spc3OD_eiIAuc | y=3.0 pred=3.269 c

In [ ]:
# 加载已保存权重，输入逐帧 `frets` / `strings`，得到整首预测与逐音符 difficult_level
# 依赖上一格已执行（定义 `difficulty_from_frets_strings`）

frets_example = [
    [0, 2, 0, 0],
    [0, 2, 3, 0],
    [2, 0, 0, 0],
]
strings_example = [
    [1],
    [1, 2],
    [0],
]

song_diff, per_note_diff = difficulty_from_frets_strings(frets_example, strings_example)
print("整首歌预测难度:", song_diff)
print("逐音符 difficult_level (与整首预测同量纲，按逐步代价占比分配):", per_note_diff)

NameError: name 'difficulty_from_frets_strings' is not defined